# Orchid Classification - ResNet18

Pipeline completa per la classificazione di orchidee del genere *Ophrys*.

**Autore originale:** Davide Di Pierro (UNINA)
**Adattamento CLI:** Marcello Russo

---

**Flusso:** Configurazione -> Augmentation -> Normalizzazione -> Preprocessing -> Training -> Testing

Questo notebook replica esattamente `run_pipeline.py`.

## 1. Configurazione

Eseguire questa cella prima di qualsiasi altra operazione.

In [ ]:
# ============================================
# IMPORT
# ============================================
import os
import random
import shutil
import torch
from torch.utils.data import DataLoader

from app.timing import Timer
from app.preprocessing.reproducibility import set_seed

# ============================================
# CONFIGURAZIONE (stessa di run_pipeline.py)
# ============================================
CONFIG = {
    # Paths
    'raw_data_path': 'dataset/raw/',
    'augmented_data_path': 'dataset/augmented/',
    'processed_train_path': 'dataset/processed/processed_data_train.pt',
    'processed_validation_path': 'dataset/processed/processed_data_validation.pt',
    'processed_test_path': 'dataset/processed/processed_data_test.pt',
    'model_path': 'models/model.pt',
    'results_path': 'results/',

    # Classi
    'class_names': [
        'O. exaltata',
        'O. garganica',
        'O. incubacea',
        'O. majellensis',
        'O. sphegodes',
        'O. sphegodes_Palena'
    ],

    # Preprocessing
    'width': 256,
    'height': 512,
    'train_size': 0.7,
    'validation_size': 0.1,  # test = 1 - train - validation
    'batch_size_preprocess': 32,
    'num_workers_preprocess': 16,
    'should_normalize': True,

    # Training
    'batch_size_train': 8,
    'num_workers_train': 4,
    'epochs': 26,
    'learning_rate': 0.001,
    'pretrained': True,
    'use_class_weights': False,  # Disabilitato: usare augmentation bilanciata invece
    'force_cpu': False,

    # Testing
    'batch_size_test': 32,
    'num_workers_test': 8,
    'sliding_window_size': 30,
    'sliding_window_stride': 15,

    # Riproducibilita
    'seed': 57,
}

# Alias per compatibilita
CLASS_NAMES = CONFIG['class_names']
CLASS_SIZE = len(CLASS_NAMES)

# ============================================
# RIPRODUCIBILITA
# ============================================
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
os.environ["PYTHONHASHSEED"] = "0"
torch.use_deterministic_algorithms(True)
set_seed(CONFIG['seed'])

# ============================================
# NORMALIZZAZIONE (calcolata sul dataset)
# ============================================
MEAN = '0.538217306137085 0.5433580875396729 0.368904173374176'
STD = '0.20864470303058624 0.23085597157478333 0.2029736042022705'

# ============================================
# DEVICE SELECTION (MPS > CUDA > CPU)
# ============================================
def get_device(force_cpu=False):
    if force_cpu:
        return torch.device('cpu')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    elif torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

print(f"Configurazione caricata. Classi: {CLASS_SIZE}")
print(f"Device: {get_device(CONFIG['force_cpu'])}")

---

## 2. Preprocessing

### 2.1 Augmentation

Divide le immagini in train/validation/test e applica data augmentation bilanciata.

In [ ]:
from app.preprocess_data import DynamicAugmentationWrapper
from app.preprocessing import rotation

if __name__ == '__main__':
    with Timer('Augmentation'):
        work_dir = 'work'
        augmentations = [
            rotation.identity,
            rotation.horizontal_and_vertical_flip,
            rotation.horizontal_flip,
            rotation.vertical_flip
        ]
        
        DynamicAugmentationWrapper(
            CONFIG['raw_data_path'],
            work_dir,
            CONFIG['augmented_data_path'],
            CONFIG['class_names'],
            CONFIG['train_size'],
            CONFIG['validation_size'],
            CONFIG['width'],
            CONFIG['height'],
            False,  # dynamic_augmentation
            augmentations,
            balanced=True  # Bilancia le classi
        )
        
        if os.path.exists(work_dir):
            shutil.rmtree(work_dir)
            print(f'Removed directory {work_dir}')
        
        print('Augmentation completata')

### 2.2 Calcolo MEAN e STD

Ricalcola i valori di normalizzazione sul dataset augmentato. Aggiornare i valori nella cella di configurazione se necessario.

In [ ]:
from app.preprocess_data import fresh_normalization

if __name__ == '__main__':
    with Timer('Normalization'):
        mean, std = fresh_normalization(
            CONFIG['augmented_data_path'] + 'train/',
            CONFIG['width'],
            CONFIG['height'],
            CONFIG['batch_size_preprocess'],
            CONFIG['num_workers_preprocess'],
            CONFIG['seed']
        )
        MEAN = ' '.join([str(x) for x in mean])
        STD = ' '.join([str(x) for x in std])
        print(f'MEAN: {MEAN}')
        print(f'STD: {STD}')
        print('\nAggiorna i valori nella cella di configurazione se diversi!')

### 2.3 Conversione in Tensori

Converte le immagini in tensori e salva su disco.

In [ ]:
from app.preprocess_data import preprocessData

PATHS = [
    CONFIG['augmented_data_path'] + 'train/',
    CONFIG['augmented_data_path'] + 'validation/',
    CONFIG['augmented_data_path'] + 'test/'
]

if __name__ == '__main__':
    with Timer('Preprocessing'):
        print('Preprocessing training dataset...')
        preprocessData(
            PATHS[0], CONFIG['processed_train_path'],
            CONFIG['width'], CONFIG['height'],
            CONFIG['batch_size_preprocess'],
            CONFIG['num_workers_preprocess'],
            CONFIG['should_normalize'], CONFIG['seed'],
            MEAN, STD
        )
        
        print('Preprocessing validation dataset...')
        preprocessData(
            PATHS[1], CONFIG['processed_validation_path'],
            CONFIG['width'], CONFIG['height'],
            CONFIG['batch_size_preprocess'],
            CONFIG['num_workers_preprocess'],
            CONFIG['should_normalize'], CONFIG['seed'],
            MEAN, STD
        )
        
        print('Preprocessing test dataset...')
        preprocessData(
            PATHS[2], CONFIG['processed_test_path'],
            CONFIG['width'], CONFIG['height'],
            CONFIG['batch_size_preprocess'],
            CONFIG['num_workers_preprocess'],
            CONFIG['should_normalize'], CONFIG['seed'],
            MEAN, STD
        )
        
        print('Preprocessing completato')

---

## 3. Training

In [ ]:
from app.train_model import loadAndTrain, saveModel

if __name__ == '__main__':
    with Timer('Training'):
        device = get_device(CONFIG['force_cpu'])
        
        if device == torch.device('cpu'):
            print('WARNING: Running on CPU. Training will be slow.')
        
        print(f'Device: {device}')
        print(f'Class weights: {CONFIG["use_class_weights"]}')
        print(f'Epochs: {CONFIG["epochs"]}')
        print(f'Batch size: {CONFIG["batch_size_train"]}')
        print(f'Learning rate: {CONFIG["learning_rate"]}')
        
        model = loadAndTrain(
            device, CLASS_SIZE,
            CONFIG['processed_train_path'],
            CONFIG['processed_validation_path'],
            CONFIG['batch_size_train'],
            CONFIG['epochs'],
            CONFIG['num_workers_train'],
            CONFIG['learning_rate'],
            CONFIG['pretrained'],
            CONFIG['model_path'],
            CONFIG['seed'],
            CONFIG['use_class_weights']
        )
        
        saveModel(model, CONFIG['model_path'])
        print('Training completato')

---

## 4. Testing

In [ ]:
from app.test_model import deviceLoader, modelLoader, testModel, generateOutputImages
from app.preprocessing.dataset_tool import getDatasetFromFile

if __name__ == '__main__':
    with Timer('Testing'):
        device = deviceLoader()
        model = modelLoader(CONFIG['model_path'], CLASS_SIZE, device)
        
        testDataset = getDatasetFromFile(CONFIG['processed_test_path'])
        testDataLoader = DataLoader(
            testDataset,
            CONFIG['batch_size_test'],
            shuffle=True,
            num_workers=CONFIG['num_workers_test'],
            prefetch_factor=2,
            persistent_workers=True
        )
        
        testModel(model, testDataLoader, device, CLASS_NAMES, CLASS_SIZE)
        
        generateOutputImages(
            testDataset, model, device,
            CLASS_NAMES, CONFIG['results_path'],
            CONFIG['sliding_window_size'],
            CONFIG['sliding_window_stride']
        )
        
        print('Testing completato')

---

## 5. Inference

Inferenza su nuove immagini non etichettate.

### 5.1 Preprocessing immagini

In [ ]:
from app.preprocessing.dataset_tool import augmentDataPath
from app.preprocessing import rotation

PATH_TO_PREPROCESS = 'inference/raw/'
INFERENCE_PATH = 'inference/test/'

if __name__ == '__main__':
    with Timer('Inference Preprocessing'):
        augmentations = [rotation.identity]
        
        if os.path.exists(PATH_TO_PREPROCESS):
            if any(os.path.isdir(os.path.join(PATH_TO_PREPROCESS, f)) 
                   for f in os.listdir(PATH_TO_PREPROCESS)):
                for folder in os.listdir(PATH_TO_PREPROCESS):
                    folder_path = os.path.join(PATH_TO_PREPROCESS, folder)
                    if os.path.isdir(folder_path):
                        augmentDataPath(
                            folder_path,
                            INFERENCE_PATH + folder + '/',
                            100000, CONFIG['width'], CONFIG['height'],
                            augmentations
                        )
            else:
                augmentDataPath(
                    PATH_TO_PREPROCESS,
                    INFERENCE_PATH,
                    100000, CONFIG['width'], CONFIG['height'],
                    augmentations
                )
            print('Preprocessing inference completato')
        else:
            print(f'Cartella {PATH_TO_PREPROCESS} non trovata')

### 5.2 Inferenza

In [ ]:
from app.inference import loadDevice, loadModel, generateOutputImages
from app.preprocess_data import getTransforms
from app.preprocessing.dataset_tool import SingleFolderDataset

DESTINATION_PATH_INFERENCE = 'inference/results/'

if __name__ == '__main__':
    with Timer('Inference'):
        device = loadDevice(CONFIG['force_cpu'])
        model = loadModel(CONFIG['model_path'], CLASS_SIZE, device)
        model.eval()
        
        test_dataset = SingleFolderDataset(
            INFERENCE_PATH,
            transform=getTransforms(CONFIG['width'], CONFIG['height'], CONFIG['should_normalize'], MEAN, STD)
        )
        
        generateOutputImages(
            test_dataset, model, device,
            CLASS_NAMES, DESTINATION_PATH_INFERENCE,
            CONFIG['sliding_window_size'],
            CONFIG['sliding_window_stride']
        )
        
        print('Inference completata')